In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import OrdinalEncoder
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.mixture import GaussianMixture
from pyod.models.iforest import IForest
from pyod.models.knn import KNN
from pyod.models.mad import MAD
import torch
import torch.nn as nn
import torch.optim as optim
pd.set_option('display.max_columns', None)

In [ ]:
# PyTorch AutoEncoder Classes
class TorchAutoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, latent_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.ReLU(),
            nn.Linear(64, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

class AutoEncoderOutlier:
    def __init__(self, latent_dim=16, lr=1e-3, epochs=30, batch_size=256, contamination=0.05):
        self.latent_dim = latent_dim
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size
        self.contamination = contamination
        self.model = None
        self.loss_fn = nn.MSELoss()

    def fit(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        dataset = torch.utils.data.TensorDataset(X)
        loader = torch.utils.data.DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        input_dim = X.shape[1]
        self.model = TorchAutoencoder(input_dim=input_dim, latent_dim=self.latent_dim)
        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        self.model.train()
        for epoch in range(self.epochs):
            for batch, in loader:
                optimizer.zero_grad()
                recon = self.model(batch)
                loss = self.loss_fn(recon, batch)
                loss.backward()
                optimizer.step()
        return self

    def reconstruction_error(self, X):
        self.model.eval()
        X = torch.tensor(X, dtype=torch.float32)
        with torch.no_grad():
            recon = self.model(X)
        errors = ((X - recon) ** 2).mean(axis=1).numpy()
        return errors

    def predict(self, X):
        errors = self.reconstruction_error(X)
        threshold = np.percentile(errors, 100 * (1 - self.contamination))
        return (errors > threshold).astype(int)

In [ ]:
# Load dataset and visualize
diamonds = sns.load_dataset("diamonds")
sns.pairplot(diamonds.sample(500), hue="price")
plt.tight_layout()
plt.show()

In [ ]:
# Encode categorical features
df = diamonds.copy()
cats = df.select_dtypes(include="category").columns.tolist()
oe = OrdinalEncoder()
df[cats] = oe.fit_transform(df[cats])

X = df.drop("price", axis=1).values
y = df["price"]

In [ ]:
# Outlier detection methods
# MAD, IQR, Z-score
mad = MAD().fit(y.values.reshape(-1,1))
df["outlier_mad"] = mad.labels_

Q1, Q3 = y.quantile(0.25), y.quantile(0.75)
IQR = Q3 - Q1
df["outlier_iqr"] = ((y < Q1 - 1.5*IQR) | (y > Q3 + 1.5*IQR)).astype(int)

z = (y - y.mean()) / y.std()
df["outlier_zscore"] = (z.abs() > 3).astype(int)

In [ ]:
# Isolation Forest
iforest = IForest(random_state=42, n_estimators=500, max_samples=1000)
iforest.fit(X)
df["outlier_iforest"] = iforest.labels_

In [ ]:
# LOF
lof = LocalOutlierFactor(n_neighbors=20, contamination='auto')
df["outlier_lof"] = (lof.fit_predict(X) == -1).astype(int)

In [ ]:
# KNN (PyOD)
knn = KNN(method='largest', n_neighbors=20)
knn.fit(X)
df["outlier_knn"] = knn.labels_

In [ ]:
# One-Class SVM
ocsvm = OneClassSVM(gamma='scale', nu=0.05)
df["outlier_ocsvm"] = (ocsvm.fit_predict(X) == -1).astype(int)

In [ ]:
# Gaussian Mixture
gmm = GaussianMixture(n_components=3, random_state=42)
gmm.fit(X)
probs = gmm.score_samples(X)
df["outlier_gmm"] = (probs < np.percentile(probs, 3)).astype(int)

In [ ]:
# PyTorch AutoEncoder
torch_ae = AutoEncoderOutlier(latent_dim=16, epochs=25, contamination=0.05)
torch_ae.fit(X)
df["outlier_torch_ae"] = torch_ae.predict(X)

In [ ]:
# Summary and visualization
print(df.filter(like='outlier').sum())
print(df.describe())
sns.pairplot(df.sample(500), hue='outlier_torch_ae')
plt.tight_layout()
plt.show()